In [3]:
import os
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from io import BytesIO
import pandas as pd

In [4]:
# Load the BLIP model and processor from Hugging Face, do this once at the start
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
def generate_image_captions(image_folder, sample_size=50):
    """
    Takes a folder of images, generates captions for a random sample, and returns a list of captions.
    """
    captions = []
    image_files = os.listdir(image_folder)[:sample_size]  # Limit to sample size
    
    for file in image_files:
        #print(file)
        raw_image = Image.open(os.path.join(image_folder, file)).convert("RGB")
        
        inputs = processor(raw_image, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        
        captions.append((caption, file))
        
    return captions

In [8]:
test = generate_image_captions("./e-commerce/Apparel/Boys/Images/images_with_product_ids")
print(test)

4203.jpg
38491.jpg
24906.jpg
24912.jpg
36325.jpg
39002.jpg
36319.jpg
39758.jpg
34056.jpg
34042.jpg
31112.jpg
40933.jpg
31106.jpg
40927.jpg
5486.jpg
31099.jpg
3815.jpg
36721.jpg
38283.jpg
39837.jpg
41754.jpg
41967.jpg
37173.jpg
37167.jpg
37601.jpg
37600.jpg
37172.jpg
35995.jpg
41966.jpg
38282.jpg
24873.jpg
37199.jpg
36720.jpg
3814.jpg
31098.jpg
40926.jpg
31107.jpg
31113.jpg
34043.jpg
34057.jpg
46833.jpg
39759.jpg
33260.jpg
36318.jpg
39003.jpg
36324.jpg
24913.jpg
38490.jpg
24907.jpg
4202.jpg
[("a boy ' s blue striped t - shirt with a sail and a sail", '4203.jpg'), ('t - shirt with print', '38491.jpg'), ('t - shirt with print', '24906.jpg'), ('t - shirt with cat print', '24912.jpg'), ('t - shirt be happy', '36325.jpg'), ('a black t - shirt with a green star on the front', '39002.jpg'), ('t - shirt with a cartoon character', '36319.jpg'), ('a picture of a grey cargo shorts', '39758.jpg'), ('a picture of a denim shorts with a skull on the side', '34056.jpg'), ('a red and white shirt with a 

In [9]:
# for handling URLs instead of local files:
def generate_image_url_captions(df, url_column, sample_size=50):
    """
    Takes a pandas DataFrame and a column of image URLs, fetches a random sample,
    and returns a list of generated captions.
    """
    captions = []
    valid_df = df.dropna(subset=[url_column])
    sample_df = valid_df.sample(n=min(sample_size, len(valid_df)))
    
    for index, row in sample_df.iterrows():
        url = row[url_column]
        
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            raw_image = Image.open(BytesIO(response.content)).convert('RGB')
            
            inputs = processor(raw_image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)
            
            captions.append((caption, url))
        except Exception as e:
            print(f"Skipped image at index {index} due to error: {e}")
            continue
        
    return captions

In [10]:
# example usage:
amazon_df = pd.read_csv("./amazon.csv")
captions = generate_image_url_captions(df=amazon_df, url_column="img_link", sample_size=10)

Skipped image at index 693 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/41IZ3JvOvwL._SX300_SY300_QL70_FMwebp_.jpg
Skipped image at index 1275 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51IE+nI0KGL._SY300_SX300_.jpg
Skipped image at index 811 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51o0rLZiIjL._SX300_SY300_QL70_FMwebp_.jpg
Skipped image at index 1062 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/410H+3lohIL._SX300_SY300_.jpg


In [11]:
print(captions)

[('a usb cable with a usb cable attached to it', 'https://m.media-amazon.com/images/I/41VKU5Lkg3L._SX300_SY300_QL70_FMwebp_.jpg'), ('the new tv from kdv', 'https://m.media-amazon.com/images/I/41YDz0uQZaL._SY300_SX300_QL70_FMwebp_.jpg'), ('lg smart tv 43 inch 4k650', 'https://m.media-amazon.com/images/I/512qfz0MI0L._SX300_SY300_QL70_FMwebp_.jpg'), ('a vacuum cleaner with a hose and a hose', 'https://m.media-amazon.com/images/I/41zqeckaQtS._SY300_SX300_QL70_FMwebp_.jpg'), ('a large, portable, infrared infrared heater', 'https://m.media-amazon.com/images/I/41WPlte6OmL._SY300_SX300_QL70_FMwebp_.jpg'), ('a black water dispe with a white lid', 'https://m.media-amazon.com/images/I/319t03ZuOML._SX300_SY300_QL70_FMwebp_.jpg')]


### next steps: 
##### test other image caption models
##### build content and semantic profiler
##### deal with text
##### automatically identify image columns